# Generate Corpora

## Load Input Words

In [ ]:
# Open Input Words
with open("input_words.txt", "r", encoding="utf-16") as f:
    
    input_words = [line.strip() for line in f if line.strip()]
    
input_words = [w.lower() for w in input_words] # normalization


## Load Small Model

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


class SmallLLM:
    def __init__(self, model_name="meta-llama/Llama-3.2-1B-Instruct", prompt_template=""):
        self.model_name = model_name
        self.prompt_template = prompt_template

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map="auto",
            dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
        self.model.eval()

    def create_prompt(self, text, prompt_template=None):
        tmpl = self.prompt_template if prompt_template is None else prompt_template

        if isinstance(text, list):
            text = ", ".join(text)

        return tmpl.replace("{text}", text)

    def format_chat(self, prompt):
        # Llama 3 expects chat format
        messages = [
            {"role": "user", "content": prompt}
        ]

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    @torch.no_grad()
    def generate(
        self,
        text,
        max_new_tokens=300,
        temperature=0.3,
        top_p=0.9,
        do_sample=False,
        prompt_template=None
    ):
        prompt = self.create_prompt(text, prompt_template)
        formatted_prompt = self.format_chat(prompt)

        inputs = self.tokenizer(
            formatted_prompt,
            return_tensors="pt",
            padding=True
        )

        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=self.tokenizer.eos_token_id
        )

        new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### CUDA Optimize

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

## Generate semantic triplets and/or sentences from the word list for Word2Vec

In [4]:
prompt_template = """
You are generating simple semantic data for a language modeling pipeline.

Given the following list of words:
{text}

For EACH word, generate exactly 3 relational triplets and 2 short sentence for each triplet.

Rules:
- Use the given word as the subject in every triplet
- Triplet format: (subject, relation, object)
- Use short relation labels from: 
    [is-a, has-property, associated-with, used-for, part-of]
- Sentences should be short, clear, and directly reflect the triplet
- Do NOT include explanations
- Do NOT skip any words
- Keep formatting EXACTLY as shown below

Output format:

WORD: <word>
TRIPLET: (<subject>, <relation>, <object>)
TRIPLET: (<subject>, <relation>, <object>)
TRIPLET: (<subject>, <relation>, <object>)
SENTENCE: <sentence>
SENTENCE: <sentence>

Repeat this block for EACH word in the list.
"""

In [5]:
prompt_template = """
Generate 27 short semantic sentences about the following word:
{text}

Example output for the word "black":
- Black is the absence of white.
- Black is a color
- Black is related to oil
- Black is often associated with darkness.
- Black is a snyonym for "dark".
"""

In [6]:
# Run: `hf auth login` this in Bash

In [ ]:
from huggingface_hub import login

# login(token = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxx")

In [ ]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"

llm = SmallLLM(model_name=model_name, prompt_template=prompt_template)

# llm_output = llm.generate(input_words, max_new_tokens=2000, do_sample=True, temperature=0.7, top_p=0.9)

llm_output = []

for word in input_words:
    output = llm.generate(word, max_new_tokens=500, do_sample=True, temperature=0.7, top_p=0.9)
    llm_output.append(output)
    # print(f"WORD: {word}\n{output}\n{'-'*40}\n")



### Preprocessing

In [ ]:
import re

def clean_llm_output(text):
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Remove headers like "Here are ..."
        if line.lower().startswith("here are"):
            continue

        # Remove lines like "WORD: ..."
        if line.lower().startswith("word:"):
            continue

        # Remove numbering like "1. ", "2. ", etc.
        line = re.sub(r"^\d+\.\s*", "", line)

        # Remove bullet points like "- ", "* "
        line = re.sub(r"^[-*]\s*", "", line)

        # Optional: remove quotes
        line = line.strip('"')

        # Only keep lines that look like sentences
        if len(line.split()) >= 3:
            cleaned.append(line)

    return cleaned

# Clean the LLM output
cleaned_output = []
for output in llm_output:
    cleaned = clean_llm_output(output)
    cleaned_output.extend(cleaned)
    
llm_output = cleaned_output

### Save Corpus

In [ ]:
# Save LLm output to file
with open("llm_text.txt", "w", encoding="utf-16") as f:
    # if list, join with newlines
    if isinstance(llm_output, list):
        f.write("\n\n".join(llm_output))
    else:
        f.write(llm_output)